# Planars — {{LANG_ID}} Annotation Status

Shows the current span results for **{{LANG_ID}}** based on filled annotation sheets.

**To use:** From the menu: **Runtime → Run all** — Google will ask for permission once.

In [ ]:
#@title Setup
import subprocess, sys, importlib, importlib.metadata

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--pre',
     'planars', 'gspread', 'google-auth'])
importlib.invalidate_caches()

from google.colab import auth
auth.authenticate_user()

import json, gspread
from google.auth import default
from googleapiclient.discovery import build

creds, _ = default(scopes=[
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets',
])
gc = gspread.authorize(creds)
drive_svc = build('drive', 'v3', credentials=creds)

# CONFIG_FILE_ID and LANG_ID are baked in at notebook generation time.
# To update them, re-run: python -m coding generate-notebooks --apply
CONFIG_FILE_ID = '{{CONFIG_FILE_ID}}'
LANG_ID = '{{LANG_ID}}'

config = json.loads(drive_svc.files().get_media(fileId=CONFIG_FILE_ID).execute())
manifest = {LANG_ID: config[LANG_ID]}

print(f'Ready ✓  ({importlib.metadata.version("planars")})')

In [ ]:
#@title Helper
from planars.io import load_filled_sheet

def show_class_reports(gc, manifest, class_name, derive_fn, format_fn):
    """Print format_result() output for every construction of class_name."""
    found_any = False
    for lang_id, lang_data in manifest.items():
        sheet_info = lang_data.get('sheets', {}).get(class_name)
        if not sheet_info:
            continue
        try:
            ss = gc.open_by_key(sheet_info['spreadsheet_id'])
        except Exception as e:
            print(f'  WARNING: could not open {class_name} sheet for {lang_id}: {e}')
            continue
        construction_params = sheet_info.get('construction_params', {})
        for construction in sheet_info['constructions']:
            found_any = True
            sep = '\u2500' * 60
            print(f'\n{sep}')
            print(f'  {class_name}  [{lang_id}]  \u2014  {construction}')
            print(sep)
            try:
                ws = ss.worksheet(construction)
            except Exception:
                print(f'  WARNING: tab not found')
                continue
            required = set(construction_params.get(construction, {}).get('param_names', []))
            loaded = load_filled_sheet(ws, required_params=required, strict=False)
            result = derive_fn(_data=loaded, strict=False)
            print(format_fn(result))
    if not found_any:
        print(f'No {class_name} sheets found in manifest.')

In [ ]:
# __CLASS_CELLS_MARKER__


---
## Domain Chart

All spans plotted as horizontal segments. Hover over a segment to see position names and span size.

In [ ]:
#@title Domain Chart
from planars.charts import collect_all_spans_from_sheets, domain_chart

df, lang_meta = collect_all_spans_from_sheets(gc, manifest)
meta = lang_meta[LANG_ID]
print(f'Done. {len(df)} spans computed.')
domain_chart(df, meta['keystone_pos'], meta['pos_to_name'])